<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-27.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 27 - Pig y Hive (Introducción)

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Simular operaciones de **Pig Latin** con Pandas (LOAD, FILTER, GROUP, JOIN, STORE)
- Simular consultas **HiveQL** con SQLite
- Comparar ambos enfoques para el procesamiento de datos

## 1. Preparación del entorno y datos de ejemplo

Creamos datasets de ejemplo que simulan datos almacenados en HDFS.

In [ ]:
import pandas as pd
import sqlite3
import random
from collections import defaultdict

random.seed(42)

categorias = ['Electrónica', 'Ropa', 'Hogar', 'Deportes', 'Libros']
ciudades = ['Madrid', 'Barcelona', 'Valencia', 'Sevilla', 'Bilbao']
productos = {
    'Electrónica': ['Portátil', 'Móvil', 'Tablet', 'Auriculares', 'Monitor'],
    'Ropa': ['Camiseta', 'Pantalón', 'Chaqueta', 'Zapatos', 'Bufanda'],
    'Hogar': ['Lámpara', 'Silla', 'Mesa', 'Alfombra', 'Cortinas'],
    'Deportes': ['Balón', 'Raqueta', 'Bicicleta', 'Mancuernas', 'Esterilla'],
    'Libros': ['Novela', 'Manual técnico', 'Cómic', 'Biografía', 'Ensayo']
}

n_ventas = 500
datos_ventas = []
for i in range(n_ventas):
    cat = random.choice(categorias)
    prod = random.choice(productos[cat])
    datos_ventas.append({
        'id': i + 1,
        'producto': prod,
        'categoria': cat,
        'importe': round(random.uniform(5.0, 500.0), 2),
        'ciudad': random.choice(ciudades),
        'fecha': f'2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}'
    })

ventas_df = pd.DataFrame(datos_ventas)

n_clientes = 100
datos_clientes = []
for i in range(n_clientes):
    datos_clientes.append({
        'cliente_id': i + 1,
        'nombre': f'Cliente_{i+1}',
        'ciudad': random.choice(ciudades),
        'tipo': random.choice(['Premium', 'Estándar', 'Básico'])
    })

clientes_df = pd.DataFrame(datos_clientes)

ventas_df['cliente_id'] = [random.randint(1, n_clientes) for _ in range(n_ventas)]

print("=== DATOS DE VENTAS (primeras 10 filas) ===")
print(ventas_df.head(10).to_string(index=False))
print(f"\nTotal de registros de ventas: {len(ventas_df)}")
print(f"\n=== DATOS DE CLIENTES (primeros 5) ===")
print(clientes_df.head().to_string(index=False))
print(f"\nTotal de clientes: {len(clientes_df)}")

## 2. Simulando operaciones Pig Latin con Pandas

En Pig Latin escribimos flujos de datos. Cada operación transforma una **relación** (similar a un DataFrame).

### Operación LOAD
```pig
ventas = LOAD 'ventas.csv' USING PigStorage(',')
    AS (id:int, producto:chararray, categoria:chararray, importe:float, ciudad:chararray, fecha:chararray);
```

In [ ]:
# === SIMULACIÓN PIG LATIN: LOAD ===
# En Pig: ventas = LOAD 'ventas.csv' USING PigStorage(',') AS (...);
# En Pandas:
ventas_df.to_csv('ventas_simuladas.csv', index=False)
ventas = pd.read_csv('ventas_simuladas.csv')
print("PIG: ventas = LOAD 'ventas.csv' ...")
print(f"Relación cargada: {len(ventas)} tuplas, {len(ventas.columns)} campos")
print(f"Schema: {list(ventas.dtypes.items())}")

# === SIMULACIÓN PIG LATIN: FILTER ===
# En Pig: grandes = FILTER ventas BY importe > 100.0;
grandes = ventas[ventas['importe'] > 100.0].copy()
print(f"\nPIG: grandes = FILTER ventas BY importe > 100.0;")
print(f"Resultado: {len(grandes)} tuplas (de {len(ventas)} originales)")

# === SIMULACIÓN PIG LATIN: FOREACH ... GENERATE ===
# En Pig: seleccion = FOREACH grandes GENERATE producto, categoria, importe;
seleccion = grandes[['producto', 'categoria', 'importe']]
print(f"\nPIG: seleccion = FOREACH grandes GENERATE producto, categoria, importe;")
print(seleccion.head().to_string(index=False))

In [ ]:
# === SIMULACIÓN PIG LATIN: GROUP ===
# En Pig: por_cat = GROUP grandes BY categoria;
por_cat = grandes.groupby('categoria')
print("PIG: por_cat = GROUP grandes BY categoria;")
print("\nGrupos formados:")
for nombre, grupo in por_cat:
    print(f"  Grupo '{nombre}': {len(grupo)} tuplas")

# === SIMULACIÓN PIG LATIN: FOREACH con agregaciones ===
# En Pig:
# totales = FOREACH por_cat GENERATE
#     group AS categoria,
#     COUNT(grandes) AS num_ventas,
#     SUM(grandes.importe) AS total,
#     AVG(grandes.importe) AS media;
totales = por_cat.agg(
    num_ventas=('importe', 'count'),
    total_importe=('importe', 'sum'),
    media_importe=('importe', 'mean')
).reset_index()

print("\nPIG: totales = FOREACH por_cat GENERATE group, COUNT, SUM, AVG;")
print(totales.to_string(index=False))

# === SIMULACIÓN PIG LATIN: ORDER ===
# En Pig: ordenado = ORDER totales BY total_importe DESC;
ordenado = totales.sort_values('total_importe', ascending=False)
print("\nPIG: ordenado = ORDER totales BY total_importe DESC;")
print(ordenado.to_string(index=False))

# === SIMULACIÓN PIG LATIN: STORE ===
# En Pig: STORE ordenado INTO 'resultados/';
ordenado.to_csv('resultado_pig_simulado.csv', index=False)
print("\nPIG: STORE ordenado INTO 'resultados/';")
print("Archivo guardado: resultado_pig_simulado.csv")

In [ ]:
# === SIMULACIÓN PIG LATIN: JOIN ===
# En Pig: combinado = JOIN ventas BY cliente_id, clientes BY cliente_id;
print("PIG: combinado = JOIN ventas BY cliente_id, clientes BY cliente_id;")
combinado = pd.merge(ventas, clientes_df, on='cliente_id', suffixes=('_venta', '_cliente'))
print(f"Resultado del JOIN: {len(combinado)} tuplas")
print(combinado[['id', 'producto', 'importe', 'nombre', 'tipo']].head(10).to_string(index=False))

# === SIMULACIÓN PIG LATIN: GROUP tras JOIN ===
# En Pig:
# por_tipo = GROUP combinado BY tipo;
# resumen = FOREACH por_tipo GENERATE
#     group AS tipo_cliente,
#     COUNT(combinado) AS compras,
#     SUM(combinado.importe) AS gasto_total;
print("\n--- Análisis por tipo de cliente ---")
resumen_tipo = combinado.groupby('tipo').agg(
    compras=('id', 'count'),
    gasto_total=('importe', 'sum'),
    gasto_medio=('importe', 'mean')
).reset_index()
print(resumen_tipo.to_string(index=False))

# === SIMULACIÓN PIG LATIN: DISTINCT ===
# En Pig: ciudades_unicas = DISTINCT (FOREACH ventas GENERATE ciudad);
ciudades_unicas = ventas[['ciudad']].drop_duplicates()
print(f"\nPIG: DISTINCT ciudades → {len(ciudades_unicas)} ciudades únicas")
print(ciudades_unicas.to_string(index=False))

## 3. Simulando HiveQL con SQLite

HiveQL es muy similar a SQL estándar. Usamos SQLite como motor para ejecutar consultas equivalentes a las que haríamos en Hive.

In [ ]:
conn = sqlite3.connect(':memory:')

# Equivalente a CREATE EXTERNAL TABLE en Hive
ventas_df.to_sql('ventas', conn, index=False, if_exists='replace')
clientes_df.to_sql('clientes', conn, index=False, if_exists='replace')

print("=== HIVEQL: DESCRIBE TABLE ===")
schema = pd.read_sql("PRAGMA table_info(ventas)", conn)
print(schema[['name', 'type']].to_string(index=False))

# Consulta básica: SELECT con WHERE
print("\n=== HIVEQL: SELECT con WHERE ===")
query1 = """
SELECT producto, categoria, importe, ciudad
FROM ventas
WHERE importe > 300
ORDER BY importe DESC
LIMIT 10;
"""
print(f"Query: {query1.strip()}")
resultado1 = pd.read_sql(query1, conn)
print(resultado1.to_string(index=False))

In [ ]:
# Consulta con GROUP BY y funciones de agregación
print("=== HIVEQL: GROUP BY con agregaciones ===")
query2 = """
SELECT
    categoria,
    COUNT(*) AS num_ventas,
    ROUND(SUM(importe), 2) AS total,
    ROUND(AVG(importe), 2) AS media,
    ROUND(MIN(importe), 2) AS minimo,
    ROUND(MAX(importe), 2) AS maximo
FROM ventas
GROUP BY categoria
ORDER BY total DESC;
"""
resultado2 = pd.read_sql(query2, conn)
print(resultado2.to_string(index=False))

# Consulta con JOIN (equivalente a JOIN en Hive)
print("\n=== HIVEQL: JOIN entre tablas ===")
query3 = """
SELECT
    c.tipo,
    COUNT(*) AS compras,
    ROUND(SUM(v.importe), 2) AS gasto_total,
    ROUND(AVG(v.importe), 2) AS gasto_medio
FROM ventas v
JOIN clientes c ON v.cliente_id = c.cliente_id
GROUP BY c.tipo
ORDER BY gasto_total DESC;
"""
resultado3 = pd.read_sql(query3, conn)
print(resultado3.to_string(index=False))

# Consulta con subconsulta (frecuente en Hive)
print("\n=== HIVEQL: Subconsulta ===")
query4 = """
SELECT ciudad, total_ciudad
FROM (
    SELECT ciudad, ROUND(SUM(importe), 2) AS total_ciudad
    FROM ventas
    GROUP BY ciudad
) subquery
WHERE total_ciudad > (
    SELECT AVG(importe) * 50 FROM ventas
)
ORDER BY total_ciudad DESC;
"""
resultado4 = pd.read_sql(query4, conn)
print(resultado4.to_string(index=False))

## 4. Comparativa Pig Latin vs HiveQL

Realizamos la **misma operación** con ambos enfoques para comparar.

In [ ]:
import time

print("="*65)
print("TAREA: Top 3 categorías por importe total (ventas > 50€)")
print("="*65)

# --- Enfoque Pig Latin (simulado con Pandas) ---
print("\n--- ENFOQUE PIG LATIN (Pandas) ---")
print("""
Script Pig Latin equivalente:
  ventas = LOAD 'ventas.csv' USING PigStorage(',') AS (...);
  filtrado = FILTER ventas BY importe > 50.0;
  agrupado = GROUP filtrado BY categoria;
  totales = FOREACH agrupado GENERATE
      group AS categoria,
      SUM(filtrado.importe) AS total;
  top3 = ORDER totales BY total DESC;
  top3 = LIMIT top3 3;
""")

t0 = time.time()
pig_resultado = (
    ventas_df[ventas_df['importe'] > 50.0]
    .groupby('categoria')['importe']
    .sum()
    .reset_index()
    .sort_values('importe', ascending=False)
    .head(3)
)
t_pig = time.time() - t0
print(f"Resultado (Pig/Pandas): {t_pig*1000:.2f} ms")
print(pig_resultado.to_string(index=False))

# --- Enfoque HiveQL (simulado con SQLite) ---
print("\n--- ENFOQUE HIVEQL (SQLite) ---")
query_hive = """
SELECT categoria, ROUND(SUM(importe), 2) AS total
FROM ventas
WHERE importe > 50.0
GROUP BY categoria
ORDER BY total DESC
LIMIT 3;
"""
print(f"Query HiveQL: {query_hive.strip()}")

t0 = time.time()
hive_resultado = pd.read_sql(query_hive, conn)
t_hive = time.time() - t0
print(f"\nResultado (Hive/SQLite): {t_hive*1000:.2f} ms")
print(hive_resultado.to_string(index=False))

print("\n" + "="*65)
print("CONCLUSIÓN: Misma tarea, dos filosofías distintas")
print("  - Pig: flujo de datos paso a paso (imperativo)")
print("  - Hive: consulta declarativa (qué quiero, no cómo)")
print("="*65)

## 5. Ejercicios propuestos

### Ejercicio 1
Escribe un "script Pig Latin" (simulado con Pandas) que:
1. Filtre ventas de la ciudad de Madrid
2. Agrupe por producto
3. Calcule el total y la media por producto
4. Ordene por total descendente

### Ejercicio 2
Escribe una consulta HiveQL (ejecutada con SQLite) que:
1. Haga JOIN entre ventas y clientes
2. Filtre solo clientes Premium
3. Agrupe por ciudad y muestre el gasto total
4. Ordene por gasto descendente

In [ ]:
# === EJERCICIO 1: Pig Latin simulado ===
# Tu código aquí:
# Paso 1 (FILTER): madrid = ventas_df[ventas_df['ciudad'] == 'Madrid']
# Paso 2 (GROUP):  por_prod = madrid.groupby('producto')
# Paso 3 (FOREACH): resultado = por_prod.agg(...)
# Paso 4 (ORDER):  resultado.sort_values(...)


# === EJERCICIO 2: HiveQL simulado ===
# Tu código aquí:
# query = """
# SELECT ...
# FROM ventas v JOIN clientes c ON ...
# WHERE c.tipo = 'Premium'
# GROUP BY ...
# ORDER BY ...;
# """
# pd.read_sql(query, conn)

print("Completa los ejercicios arriba descomentando y rellenando el código.")

## Recursos

- [Apache Pig - Documentación oficial](https://pig.apache.org/docs/latest/)
- [Apache Hive - Wiki](https://cwiki.apache.org/confluence/display/Hive/Home)
- [Pig Latin Reference Manual](https://pig.apache.org/docs/latest/basic.html)
- [HiveQL Language Manual](https://cwiki.apache.org/confluence/display/Hive/LanguageManual)

---

[← Sesión 26](./sesion-26.ipynb) | [Índice](../sesiones/index.md) | [Siguiente sesión →](./sesion-28.ipynb)